# OG-RAG Hypergraph - Original Algorithm


```bash
pip install -r requirements_minimal_hypergraph.txt
```

---

In [49]:
!pip install -r requirements_minimal_hypergraph.txt

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)



[notice] A new release of pip available: 22.2.1 -> 25.2
[notice] To update, run: pip install --upgrade pip


## 1. Setup & Imports

In [ ]:
import os
import sys
import yaml
import json
import numpy as np
import pandas as pd
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

print(f"Working directory: {os.getcwd()}")

Working directory: /Users/soc_041/Documents/Github/ograg2


## 2. Load API Keys

In [46]:
api_keys_file = "api_keys.yaml"

if not os.path.exists(api_keys_file):
    sample_config = {
        'GROQ_API_KEY': 'gsk_your-groq-key-here'
    }
    with open(api_keys_file, 'w') as f:
        yaml.dump(sample_config, f, default_flow_style=False)
    print("Created api_keys.yaml - Please add your Groq API key!")
else:
    with open(api_keys_file, 'r') as f:
        api_keys = yaml.safe_load(f)
    
    for key, value in api_keys.items():
        if value:
            os.environ[key] = str(value)
    
    has_groq = bool(api_keys.get('GROQ_API_KEY', '').startswith('gsk_'))
    
    print("API Keys Status:")
    print(f"  Groq: {'OK' if has_groq else 'MISSING'}")
    
    if not has_groq:
        print("\nlease update 'api_keys.yaml' with valid Groq API key")

API Keys Status:
  Groq: OK


## 3. Import Original Code + Frameworks

- query_engine.ontograph_query_engine - (HyperNode, HyperEdge, OntoHyperGraph)


In [32]:
import sys
from unittest.mock import MagicMock

# Mock azureml
sys.modules['azureml'] = MagicMock()
sys.modules['azureml.rag'] = MagicMock()
sys.modules['azureml.rag.utils'] = MagicMock()
sys.modules['azureml.rag.utils.connections'] = MagicMock()
sys.modules['langchain_together'] = MagicMock()
sys.modules['llama_index.legacy'] = MagicMock()
sys.modules['llama_index.legacy.embeddings'] = MagicMock()

import importlib.util

spec = importlib.util.spec_from_file_location(
    "ontograph_module",
    "query_engine/ontograph_query_engine.py"
)
ontograph_module = importlib.util.module_from_spec(spec)
import numpy as np

def cosine_similarity(vec1, vec2):
    """Compute cosine similarity between two vectors"""
    vec1 = np.array(vec1)
    vec2 = np.array(vec2)
    dot_product = np.dot(vec1, vec2)
    norm1 = np.linalg.norm(vec1)
    norm2 = np.linalg.norm(vec2)
    if norm1 == 0 or norm2 == 0:
        return 0.0
    return dot_product / (norm1 * norm2)

def flatten_tree(node):
    return [node] if isinstance(node, dict) else []

def load_graph_nodes(path):
    return []

def load_graph_nodes_chunks(path, chunks):
    return [], []

utils_mock = MagicMock()
utils_mock.cosine_similarity = cosine_similarity
utils_mock.flatten_tree = flatten_tree
utils_mock.load_graph_nodes = load_graph_nodes
utils_mock.load_graph_nodes_chunks = load_graph_nodes_chunks

sys.modules['utils'] = utils_mock

print("Created utils mock with necessary functions")

spec.loader.exec_module(ontograph_module)

HyperNode = ontograph_module.HyperNode
HyperEdge = ontograph_module.HyperEdge
OntoHyperGraph = ontograph_module.OntoHyperGraph
OntoHyperGraphQueryEngine = ontograph_module.OntoHyperGraphQueryEngine

print("Imported HyperNode, HyperEdge, OntoHyperGraph classes")
from langchain_openai import ChatOpenAI
from langchain_community.embeddings import HuggingFaceEmbeddings

print("   - HyperNode: Custom class from ontograph_query_engine.py")
print("   - HyperEdge: Custom class from ontograph_query_engine.py")
print("   - OntoHyperGraph: Custom class from ontograph_query_engine.py")
print("   - ChatOpenAI: LangChain (Groq-compatible)")
print("   - HuggingFaceEmbeddings: LangChain (local)")

Created utils mock with necessary functions
Imported HyperNode, HyperEdge, OntoHyperGraph classes
   - HyperNode: Custom class from ontograph_query_engine.py
   - HyperEdge: Custom class from ontograph_query_engine.py
   - OntoHyperGraph: Custom class from ontograph_query_engine.py
   - ChatOpenAI: LangChain (Groq-compatible)
   - HuggingFaceEmbeddings: LangChain (local)


## 4. Initialize LLM & Embeddings

In [33]:
MODEL_CONFIG = {
    'model': 'llama-3.3-70b-versatile',
    'temperature': 0.0,
    'max_tokens': 4096,
}

EMBEDDING_CONFIG = {
    'model_name': 'BAAI/bge-large-en-v1.5',
}

print("Initializing Groq LLM...")
llm = ChatOpenAI(
    model=MODEL_CONFIG['model'],
    api_key=os.environ['GROQ_API_KEY'],
    base_url='https://api.groq.com/openai/v1',
    temperature=MODEL_CONFIG['temperature'],
    max_tokens=MODEL_CONFIG['max_tokens'],
)
print(f"Groq LLM: {MODEL_CONFIG['model']}")

print("\nInitializing HuggingFace Embeddings...")
embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_CONFIG['model_name']
)
print(f"Embeddings: {EMBEDDING_CONFIG['model_name']}")

Initializing Groq LLM...
Groq LLM: llama-3.3-70b-versatile

Initializing HuggingFace Embeddings...
Embeddings: BAAI/bge-large-en-v1.5
Embeddings: BAAI/bge-large-en-v1.5


## 5. Load Ontology Data

In [ ]:
DATASET = 'soybean' # change to "wheat" to use other dataset
DATA_DIR = f'data/kg/{DATASET}'
ONTOLOGY_DIR = f'{DATA_DIR}/ontology'

print(f"Dataset: {DATASET.upper()}")
print(f"Ontology directory: {ONTOLOGY_DIR}")
print(f"   Exists: {'OK' if os.path.exists(ONTOLOGY_DIR) else 'None'}")

if os.path.exists(ONTOLOGY_DIR):
    ontology_files = sorted([f for f in os.listdir(ONTOLOGY_DIR) if f.endswith('.jsonld')])
    print(f"\nFound {len(ontology_files)} ontology files:")
    for f in ontology_files[:5]:
        print(f"   - {f}")
    if len(ontology_files) > 5:
        print(f"   ... and {len(ontology_files) - 5} more files")

Dataset: SOYBEAN
Ontology directory: data/kg/soybean/ontology
   Exists: OK

Found 4 ontology files:
   - ontology_node_0.jsonld
   - ontology_node_1.jsonld
   - ontology_node_2.jsonld
   - ontology_node_3.jsonld


## 6. Flatten Ontology Nodes

flatten nested structures thành key-value facts.

In [35]:
def flatten_tree(node, max_depth=3):
    """
    Converts nested JSON-LD structure thành flat key-value dictionaries.
    Mỗi dict = 1 fact (sẽ thành 1 HyperEdge)
    """
    flattened_nodes = []
    node_context = {k: str(v) for k, v in node.items() 
                   if not isinstance(v, dict) and not isinstance(v, list) 
                   and k not in ['@context', '@graph']}
    
    if node_context:
        flattened_nodes.append(node_context)
    
    for k, v in node.items():
        if k in ['@context', '@graph']:
            continue
            
        if isinstance(v, dict):
            for nested_node in flatten_tree(v, max_depth=max_depth-1):
                if max_depth > 0:
                    nested_with_context = {**node_context, **{f'{k}_{nk}': nv for nk, nv in nested_node.items()}}
                    flattened_nodes.append(nested_with_context)
                
        elif isinstance(v, list):
            # List of items
            for item in v[:10]:  # Limit to 10 items per list
                if isinstance(item, dict):
                    for nested_node in flatten_tree(item, max_depth=max_depth-1):
                        if max_depth > 0:
                            nested_with_context = {**node_context, **{f'{k}_{nk}': nv for nk, nv in nested_node.items()}}
                            flattened_nodes.append(nested_with_context)
                else:
                    flattened_nodes.append({**node_context, k: str(item)})
    
    return flattened_nodes

def load_ontology_nodes(ontology_dir):
    """Load and flatten all ontology nodes from jsonld files"""
    nodes = []
    
    for fname in sorted(os.listdir(ontology_dir)):
        if fname.endswith('.jsonld'):
            fpath = os.path.join(ontology_dir, fname)
            with open(fpath, 'r') as f:
                data = json.load(f)
                if '@graph' in data:
                    for node in data['@graph']:
                        # Flatten nested structures
                        flattened = flatten_tree(node)
                        nodes.extend(flattened)
    
    return nodes

print(" Flattening ontology nodes...")
ontology_facts = load_ontology_nodes(ONTOLOGY_DIR)
print(f" Created {len(ontology_facts)} flattened facts (will become HyperEdges)")

if ontology_facts:
    print(f"\nSample fact (future HyperEdge):")
    sample = ontology_facts[0]
    print(json.dumps(sample, indent=2)[:400] + "...")
    print(f"\n   Keys: {list(sample.keys())}")
    print(f"   Will create {len(sample)} HyperNodes (one per key-value pair)")

 Flattening ontology nodes...
 Created 153 flattened facts (will become HyperEdges)

Sample fact (future HyperEdge):
{
  "@type": "cropCult:Crop",
  "name": "Soybean",
  "seed_germination_test_requirements_are": "Farmers are advised to check germination status of seed purchased/available with them before sowing. To ensure optimum plant population and thereby good yield, minimum 70% germination is essential.",
  "harvesting_guidelines_are": "Optimum time of harvesting is very important for soybean as it causes yi...

   Keys: ['@type', 'name', 'seed_germination_test_requirements_are', 'harvesting_guidelines_are', 'storage_guidelines_are']
   Will create 5 HyperNodes (one per key-value pair)


## 7. Pre-compute Embeddings

Embed unique keys & values một lần, cache lại


In [36]:
print("Pre-computing embeddings for unique keys & values...")

unique_texts = set()
for fact in ontology_facts:
    for key, value in fact.items():
        unique_texts.add(key)
        unique_texts.add(str(value))

unique_texts = list(unique_texts)
print(f"Found {len(unique_texts)} unique texts (keys + values)")

print(f"\n Embedding {len(unique_texts)} texts...")
unique_embeddings = embeddings.embed_documents(unique_texts)

embeddings_dict = {text: np.array(emb) for text, emb in zip(unique_texts, unique_embeddings)}

print(f"\nCached {len(embeddings_dict)} embeddings")
print(f"   Embedding dimension: {len(unique_embeddings[0])}")
print(f"   Memory usage: ~{len(embeddings_dict) * len(unique_embeddings[0]) * 4 / 1024 / 1024:.1f} MB")

Pre-computing embeddings for unique keys & values...
Found 266 unique texts (keys + values)

 Embedding 266 texts...

Cached 266 embeddings
   Embedding dimension: 1024
   Memory usage: ~1.0 MB

Cached 266 embeddings
   Embedding dimension: 1024
   Memory usage: ~1.0 MB


## 8. Build Hypergraph 

1. Mỗi fact (dict) → 1 **HyperEdge**
2. Mỗi key-value pair → 1 **HyperNode** 
3. HyperNode lưu embeddings riêng cho key & value
4. HyperEdge chứa list các HyperNodes

In [37]:
hypergraph = OntoHyperGraph.from_fact_lists(
    facts=ontology_facts,
    embed_model=embeddings,
    embeddings=embeddings_dict  # Pass pre-computed embeddings
)

print(f"\nOntoHyperGraph created!")
print(f"   - {len(hypergraph.nodes)} HyperNodes (unique key-value pairs)")
print(f"   - {len(hypergraph.edges)} HyperEdges (facts)")

# Show sample HyperNode
if hypergraph.nodes:
    sample_node = hypergraph.nodes[0]
    print(f"\nSample HyperNode:")
    print(f"   Key: '{sample_node.key}'")
    print(f"   Value: '{sample_node.value}'")
    print(f"   Key embedding shape: {sample_node.key_embedding.shape if hasattr(sample_node.key_embedding, 'shape') else len(sample_node.key_embedding)}")
    print(f"   Value embedding shape: {sample_node.value_embedding.shape if hasattr(sample_node.value_embedding, 'shape') else len(sample_node.value_embedding)}")
    print(f"   Connected to {len(sample_node.edge_ids)} HyperEdges")

# Show sample HyperEdge
if hypergraph.edges:
    sample_edge = hypergraph.edges[0]
    print(f"\nSample HyperEdge:")
    print(f"   Contains {len(sample_edge.nodes)} HyperNodes:")
    for i, node in enumerate(sample_edge.nodes[:3], 1):
        print(f"      {i}. {node.key}: {node.value}")
    if len(sample_edge.nodes) > 3:
        print(f"      ... and {len(sample_edge.nodes) - 3} more nodes")
    print(f"\n   As text: {sample_edge.to_text()[:200]}...")


OntoHyperGraph created!
   - 254 HyperNodes (unique key-value pairs)
   - 153 HyperEdges (facts)

Sample HyperNode:
   Key: '@type'
   Value: 'cropCult:Crop'
   Key embedding shape: (1024,)
   Value embedding shape: (1024,)
   Connected to 95 HyperEdges

Sample HyperEdge:
   Contains 5 HyperNodes:
      1. @type: cropCult:Crop
      2. name: Soybean
      3. seed_germination_test_requirements_are: Farmers are advised to check germination status of seed purchased/available with them before sowing. To ensure optimum plant population and thereby good yield, minimum 70% germination is essential.
      ... and 2 more nodes

   As text: @type 'cropCult:Crop', name 'Soybean', seed_germination_test_requirements_are 'Farmers are advised to check germination status of seed purchased/available with them before sowing. To ensure optimum pl...


## 9. Create Query Engine + Optimize Prompt

Override prompt gốc để LLM trả lời ngắn gọn hơn

In [38]:
print("Creating OntoHyperGraphQueryEngine...")

ontograph_module.RAG_QUERY_PROMPT = """Given the context below, answer the following question CONCISELY.
Note that the context is provided as a list of valid facts in a dictionary format.

IMPORTANT: 
- Answer DIRECTLY and BRIEFLY
- For simple questions, give ONLY the answer (1-2 sentences max)
- For complex questions, provide key points in a SHORT paragraph
- Do NOT repeat the question in your answer
- Do NOT explain your reasoning process unless asked

Context: {context}

Question: {query_str}

Answer:
"""

print("Updated prompt for concise answers")

query_engine = OntoHyperGraphQueryEngine(
    llm=llm,
    onto_hypergraph=hypergraph,
    vector_retriever=None  # Optional - có thể add VectorIndexRetriever
)

print("Query engine ready!")
print("\nQuery flow:")
print("   1. Embed query")
print("   2. Find top-k HyperNodes (by key & value similarity)")
print("   3. Get HyperEdges covering those nodes")
print("   4. Format context from HyperEdges")
print("   5. Generate CONCISE answer with LLM")

Creating OntoHyperGraphQueryEngine...
Updated prompt for concise answers
Query engine ready!

Query flow:
   1. Embed query
   2. Find top-k HyperNodes (by key & value similarity)
   3. Get HyperEdges covering those nodes
   4. Format context from HyperEdges
   5. Generate CONCISE answer with LLM


## 10. Test với Câu Hỏi Đầu Tiên

In [39]:
test_question = "Which pest can be controlled with Imidacloprid 48 FS pesticide in soybean crops?"
ground_truth = "YELLOW MOSAIC VIRUS (YMV)"

print(f"Question: {test_question}")
print(f"Ground Truth: {ground_truth}")
print("\n" + "="*80)

# Query với original hypergraph algorithm
response, context = query_engine.query(
    query_str=test_question,
    top_k=5,  # Top 5 HyperEdges
    nodes_top_k=20,  # Top 20 HyperNodes
    return_context=True
)

# Extract answer
if hasattr(response, 'content'):
    answer = response.content
elif hasattr(response, 'response'):
    answer = response.response
else:
    answer = str(response)

print(f"Answer:\n{answer}")

# Check correctness
is_correct = ("ymv" in answer.lower() or 
              "yellow mosaic" in answer.lower() or
              "yellow mosaic virus" in answer.lower())

print("\n" + "="*80)
if is_correct:
    print("CORRECT! Answer matches ground truth!")
elif "no information" not in answer.lower():
    print("Answer has info but doesn't match ground truth")
else:
    print("No information found")

print("\n" + "="*80)
print("Retrieved HyperEdges (Context):")
print("-"*80)
for i, ctx in enumerate(context[:3], 1):
    print(f"\n{i}. {ctx}")
    print("-"*40)

Question: Which pest can be controlled with Imidacloprid 48 FS pesticide in soybean crops?
Ground Truth: YELLOW MOSAIC VIRUS (YMV)

Answer:
YELLOW MOSAIC VIRUS (YMV)

CORRECT! Answer matches ground truth!

Retrieved HyperEdges (Context):
--------------------------------------------------------------------------------

1. {'@type': 'cropCult:Crop', 'name': 'Soybean', 'seed_germination_test_requirements_are': 'Seeds should have a germination rate of at least 80%', 'harvesting_guidelines_are': 'Harvest when 85-90% pods have turned brown and lost their green color. The moisture content of the seeds should be around 10-12%.', 'storage_guidelines_are': 'Store in a cool, dry place. Use hermetic storage bags for long-term storage.', 'needs_pest_treatements_@type': 'cropCult:PestTreatements', 'needs_pest_treatements_pest_name': 'White Grub, Gram Pod Borer, Tobacco Caterpillar, Green Semilooper, Girdle Beetle', 'needs_pest_treatements_pest_symptoms': 'Drying of plants in linear patches, heavy yi

## 11. Debug: Cách Tính Similarity

### Hypergraph Retrieval Strategy:

**Step 1: Tính Similarity Riêng Biệt (Key vs Value)**

Mỗi HyperNode có 2 embeddings:
- `key_embedding`: Embedding của attribute name (e.g., "pesticide_name")
- `value_embedding`: Embedding của value (e.g., "Imidacloprid 48 FS")

**Similarity Methods:**
```python
# Method 1: SUM (default) - Cộng cả 2
similarity = cos(query, key_emb) + cos(query, value_emb)

# Method 2: KEY_ONLY - Chỉ key
similarity = cos(query, key_emb)

# Method 3: VALUE_ONLY - Chỉ value  
similarity = cos(query, value_emb)

# Method 4: PRODUCT - Nhân cả 2
similarity = cos(query, key_emb) * cos(query, value_emb)
```

**Step 2: Retrieve Nodes (Dual Ranking)**

Code gốc retrieve **RIÊNG BIỆT**:
```python
# Top-K nodes by KEY similarity only
key_nodes = top_k nodes sorted by cos(query, key_emb)

# Top-K nodes by VALUE similarity only  
value_nodes = top_k nodes sorted by cos(query, value_emb)

# Combine both lists
retrieved_nodes = key_nodes + value_nodes
```

**Step 3: Get Covering HyperEdges**

Từ retrieved nodes → Lấy edges chứa nodes đó

---

### Ví Dụ:

**Query:** "Which pest can be controlled with Imidacloprid?"

**HyperNode 1:**
- Key: `"pesticide_name"` → key_emb
- Value: `"Imidacloprid 48 FS"` → value_emb
- Similarity:
  - Key-only: cos(query, "pesticide_name") = **0.3** (medium)
  - Value-only: cos(query, "Imidacloprid 48 FS") = **0.9** (high!)
  - Sum: 0.3 + 0.9 = **1.2**

**HyperNode 2:**
- Key: `"pest_controlled"`  
- Value: `"YELLOW MOSAIC VIRUS (YMV)"`
- Similarity:
  - Key-only: cos(query, "pest_controlled") = **0.7** (high - "pest" match!)
  - Value-only: cos(query, "YMV") = **0.4** (medium)
  - Sum: 0.7 + 0.4 = **1.1**

**Retrieval:**
- Top by KEY: Node 2 (0.7) → Tìm được "pest_controlled"
- Top by VALUE: Node 1 (0.9) → Tìm được "Imidacloprid"
- → Combine → Get edge chứa cả 2 → Perfect answer!

---


In [48]:
print("Debug: Phân Tích Similarity Strategy")
print("="*80)

query_embedding = embeddings.embed_query(test_question)

print(f"\nQuery: {test_question}")
print("\n" + "="*80)
print("DUAL RANKING STRATEGY")
print("="*80)

# Get top nodes by KEY only
print("\n1. Top 5 HyperNodes by KEY similarity:")
print("-"*80)
key_nodes = sorted(hypergraph.nodes, 
                  key=lambda x: x.similarity(query_embedding, method='key_only'), 
                  reverse=True)[:5]

for i, node in enumerate(key_nodes, 1):
    key_sim = node.similarity(query_embedding, method='key_only')
    print(f"\n{i}. [Key Score: {key_sim:.4f}]")
    print(f"   Key: '{node.key}'")
    print(f"   Value: '{str(node.value)[:80]}{'...' if len(str(node.value)) > 80 else ''}")

# Get top nodes by VALUE only
print("\n" + "="*80)
print("2. Top 5 HyperNodes by VALUE similarity:")
print("-"*80)
value_nodes = sorted(hypergraph.nodes, 
                    key=lambda x: x.similarity(query_embedding, method='value_only'), 
                    reverse=True)[:5]

for i, node in enumerate(value_nodes, 1):
    value_sim = node.similarity(query_embedding, method='value_only')
    print(f"\n{i}. [Value Score: {value_sim:.4f}]")
    print(f"   Key: '{node.key}'")
    print(f"   Value: '{str(node.value)[:80]}{'...' if len(str(node.value)) > 80 else ''}")

# Compare với SUM method
print("\n" + "="*80)
print("3. Top 5 HyperNodes by SUM (key + value):")
print("-"*80)
sum_nodes = sorted(hypergraph.nodes, 
                  key=lambda x: x.similarity(query_embedding, method='sum'), 
                  reverse=True)[:5]

for i, node in enumerate(sum_nodes, 1):
    key_sim = node.similarity(query_embedding, method='key_only')
    value_sim = node.similarity(query_embedding, method='value_only')
    sum_sim = key_sim + value_sim
    print(f"\n{i}. [Sum: {sum_sim:.4f}] = [Key: {key_sim:.4f}] + [Value: {value_sim:.4f}]")
    print(f"   Key: '{node.key}'")
    print(f"   Value: '{str(node.value)[:80]}{'...' if len(str(node.value)) > 80 else ''}")
    

print("\n" + "="*80)
print("KEY INSIGHT:")
print("="*80)
print("   - Retrieve top-K by KEY + top-K by VALUE")
print("   - Combine cả 2 lists")
print("   - Đảm bảo không miss nodes có 1 aspect score cao")


Debug: Phân Tích Similarity Strategy

Query: Which pest can be controlled with Imidacloprid 48 FS pesticide in soybean crops?

DUAL RANKING STRATEGY

1. Top 5 HyperNodes by KEY similarity:
--------------------------------------------------------------------------------

1. [Key Score: 0.5819]
   Key: 'needs_pest_treatements_pest_control_@type'
   Value: 'cropCult:PesticideList

2. [Key Score: 0.5819]
   Key: 'needs_pest_treatements_pest_control_@type'
   Value: 'PesticideList

3. [Key Score: 0.5806]
   Key: 'needs_pest_treatements_pest_control_name'
   Value: 'Various

4. [Key Score: 0.5806]
   Key: 'needs_pest_treatements_pest_control_name'
   Value: 'Imidachloprid, Chlorpyrifos, Thichloprid, Profenophos, Betacyfluthrin + Imidaclo...

5. [Key Score: 0.5806]
   Key: 'needs_pest_treatements_pest_control_name'
   Value: 'Thiamethoxam 30 FS

2. Top 5 HyperNodes by VALUE similarity:
--------------------------------------------------------------------------------

1. [Value Score: 0.7512]
 

## 12. Batch Test với Nhiều Câu Hỏi

In [41]:
QUESTIONS_FILE = f'{DATA_DIR}/questions/ontodoc_ragas/testset_reasoning.csv'

if os.path.exists(QUESTIONS_FILE):
    questions_df = pd.read_csv(QUESTIONS_FILE)
    print(f"Loaded {len(questions_df)} test questions")
    print(f"\nColumns: {list(questions_df.columns)}")
else:
    print(f"Questions file not found: {QUESTIONS_FILE}")
   

Loaded 97 test questions

Columns: ['question', 'contexts', 'ground_truth', 'evolution_type', 'metadata', 'episode_done']


In [42]:
import time
from tqdm.auto import tqdm

NUM_QUESTIONS = min(5, len(questions_df))

print(f"🧪 Testing with {NUM_QUESTIONS} questions...")
print("="*80)

results = []

for idx in tqdm(range(NUM_QUESTIONS), desc="Processing questions"):
    question = questions_df.iloc[idx]['question']
    ground_truth = questions_df.iloc[idx].get('ground_truth', 'N/A')
    
    # Query with hypergraph
    start_time = time.time()
    response, context = query_engine.query(
        query_str=question,
        top_k=5,
        nodes_top_k=20,
        return_context=True
    )
    query_time = time.time() - start_time
    
    # Extract answer
    if hasattr(response, 'content'):
        answer = response.content
    elif hasattr(response, 'response'):
        answer = response.response
    else:
        answer = str(response)
    
    results.append({
        'question': question,
        'ground_truth': ground_truth,
        'answer': answer,
        'query_time': query_time,
        'context_length': len(str(context)),
        'num_hyperedges': len(context)
    })
    
    time.sleep(0.5)  # Rate limit

results_df = pd.DataFrame(results)
print(f"\nCompleted {len(results_df)} questions!")
print(f"\nStatistics:")
print(f"   Average query time: {results_df['query_time'].mean():.2f}s")
print(f"   Average context length: {results_df['context_length'].mean():.0f} chars")
print(f"   Average HyperEdges retrieved: {results_df['num_hyperedges'].mean():.1f}")

🧪 Testing with 5 questions...


Processing questions: 100%|██████████| 5/5 [00:06<00:00,  1.27s/it]


Completed 5 questions!

Statistics:
   Average query time: 0.76s
   Average context length: 4439 chars
   Average HyperEdges retrieved: 5.0


## 13. Display Results

In [43]:
print("Test Results:")
print("="*80)

for i, row in results_df.iterrows():
    print(f"\n{i+1}. Question: {row['question']}...")
    print(f"   Ground Truth: {row['ground_truth']}")
    print(f"   Answer: {row['answer']}...")
    print(f"   Time: {row['query_time']:.2f}s")
    print(f"   HyperEdges: {row['num_hyperedges']}")
    print("-"*80)

Test Results:

1. Question: Which pest can be controlled with Imidacloprid 48 FS pesticide in soybean crops?...
   Ground Truth: YELLOW MOSAIC VIRUS (YMV)
   Answer: YELLOW MOSAIC VIRUS (YMV)...
   Time: 0.71s
   HyperEdges: 5
--------------------------------------------------------------------------------

2. Question: What type of soil should be avoided for growing soybeans due to its medium to high nutrient holding capacity and medium water holding capacity?...
   Ground Truth: Sandy and heavy black soils
   Answer: Sandy and heavy black soils should be avoided for growing soybeans....
   Time: 0.72s
   HyperEdges: 5
--------------------------------------------------------------------------------

3. Question: What is the name of the crop that requires seeds with a germination rate of at least 80%, should be harvested when 85-90% pods have turned brown and lost their green color, and should be stored in a cool, dry place using hermetic storage bags for long-term storage?...
   Groun

## 14. Save Results

In [ ]:
print("EVALUATION: Hypergraph Performance on Soybean Dataset")
print("="*80)

# Analyze results
total = len(results_df)
print(f"\nTest Set: {total} questions")

def check_correctness(answer, ground_truth):
    """Simple check if answer contains ground truth"""
    answer_lower = answer.lower()
    gt_lower = str(ground_truth).lower()
    
    if gt_lower in answer_lower:
        return True
    
    gt_terms = [t.strip() for t in gt_lower.replace(',', ' ').split() if len(t.strip()) > 3]
    if gt_terms:
        matched_terms = sum(1 for term in gt_terms if term in answer_lower)
        return matched_terms / len(gt_terms) > 0.5
    
    return False

results_df['is_correct'] = results_df.apply(
    lambda x: check_correctness(x['answer'], x['ground_truth']), axis=1
)

correct = results_df['is_correct'].sum()
accuracy = (correct / total) * 100

print(f"\nCorrectness:")
print(f"   Correct: {correct}/{total} ({accuracy:.1f}%)")
print(f"   Incorrect: {total - correct}/{total}")

print(f"\nPerformance:")
print(f"   Average query time: {results_df['query_time'].mean():.2f}s")
print(f"   Min query time: {results_df['query_time'].min():.2f}s")
print(f"   Max query time: {results_df['query_time'].max():.2f}s")

print(f"\nContext Retrieved:")
print(f"   Average HyperEdges: {results_df['num_hyperedges'].mean():.1f}")
print(f"   Average context length: {results_df['context_length'].mean():.0f} chars")

print("\n" + "="*80)
print("KEY FINDINGS:")
print("="*80)


print("\nExample Success (Question 1):")
q1 = results_df.iloc[0]
print(f"   Q: {q1['question'][:80]}...")
print(f"   GT: {q1['ground_truth']}")
print(f"   A: {q1['answer'][:80]}...")
print(f"   Correct: {q1['is_correct']}")
print(f"   Time: {q1['query_time']:.2f}s")

EVALUATION: Hypergraph Performance on Soybean Dataset

Test Set: 5 questions

Correctness:
   Correct: 4/5 (80.0%)
   Incorrect: 1/5

⏱Performance:
   Average query time: 0.76s
   Min query time: 0.64s
   Max query time: 1.00s

Context Retrieved:
   Average HyperEdges: 5.0
   Average context length: 4439 chars

KEY FINDINGS:

Example Success (Question 1):
   Q: Which pest can be controlled with Imidacloprid 48 FS pesticide in soybean crops?...
   GT: YELLOW MOSAIC VIRUS (YMV)
   A: YELLOW MOSAIC VIRUS (YMV)...
   Correct: True
   ⏱Time: 0.71s


### Save result

In [ ]:
output_dir = 'results/hypergraph_test'
os.makedirs(output_dir, exist_ok=True)

output_json = f'{output_dir}/{DATASET}_hypergraph_results.json'
results_df.to_json(output_json, orient='records', indent=2)
print(f"Saved results to: {output_json}")

output_csv = f'{output_dir}/{DATASET}_hypergraph_results.csv'
results_df.to_csv(output_csv, index=False)
print(f"Saved results to: {output_csv}")

💾 Saved results to: results/hypergraph_test/soybean_hypergraph_results.json
💾 Saved results to: results/hypergraph_test/soybean_hypergraph_results.csv
